In [27]:
from scripts.data_loader import get_splitted_df, LabeledFaceDataset, labeled_face_dataset_collate, PairDataset
from scripts.data_loader import reverse_preprocess_input, pair_dataset_collate
from scripts.model import Arcface
from scripts.evaluate import evaluate

In [6]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

### Prepare Dataset

In [10]:
ROOT_PATH = "data/faces_webface_112x112"

In [11]:
val_batch_size = 32
val_num_workers = 4

In [12]:
agedb30_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "agedb_30.bin")

In [13]:
agedb30_loader =  DataLoader(
    agedb30_dataset, 
    shuffle=False, 
    batch_size=val_batch_size, 
    num_workers=val_num_workers, 
    pin_memory=True,
    collate_fn=pair_dataset_collate, 
)

In [14]:
# cfp_ff_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "cfp_ff.bin")
# cfp_fp_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "cfp_fp.bin")
# calfw_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "calfw.bin")
# cplfw_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "cplfw.bin")
# lfw_dataset = PairDataset(root_path = ROOT_PATH, dataset_path = "lfw.bin", images_are_bytes = True)

In [15]:
# Checking

# images_1, images_2, targets = next(iter(agedb30_loader))
# idx = 3
# reverse_preprocess_input(images_1[0])
# reverse_preprocess_input(images_2[0])

### Config

In [31]:
num_classes = 10572

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [18]:
model = Arcface(num_classes=num_classes).to(device)

In [19]:
num_classes

10572

In [20]:
sum(p.numel() for p in model.parameters())

21993280

In [21]:
model.head.weight.shape[0]

10572

### Evaluate

In [24]:
def evaluate(test_loader):
    labels, distances = [], []
    for _, (data_a, data_p, label) in enumerate(test_loader):
        with torch.no_grad():
            data_a, data_p = data_a.type(torch.FloatTensor), data_p.type(torch.FloatTensor)
            data_a, data_p = data_a.to(device), data_p.to(device)
            
            out_a, out_p = model(data_a), model(data_p)
            dists = torch.sqrt(torch.sum((out_a - out_p) ** 2, 1))
        distances.append(dists.data.cpu().numpy())
        labels.append(label.data.cpu().numpy())        
        break # Comment This
    return labels, distances

In [25]:
labels, distances = evaluate(agedb30_loader)

In [26]:
labels      = np.array([sublabel for label in labels for sublabel in label])
distances   = np.array([subdist for dist in distances for subdist in dist])

In [28]:
tpr, fpr, accuracy, val, val_std, far, best_thresholds = evaluate(distances,labels)